### Incremental Ingestion
* Open F1 API (2025 - )

In [0]:
import time
import requests
import pandas as pd 
from pyspark.sql.functions import col

In [0]:
#endpts that can be downloaded directly
metadata_endpts=["meetings","drivers","sessions","starting_grid"]

#endpoints that require parameters
session_endpts = ["session_result", "stints","pit"]


In [0]:


BASE_URL="https://api.openf1.org/v1"

def get_endpoints(endpoint,output_path,params={"csv":"true"}):
    url=f"{BASE_URL}/{endpoint}"
    response=requests.get(url,params=params)

    response.raise_for_status()

    with open(output_path,"wb") as f: #write as binary(used for bytes instead of text) to f (output file)
        f.write(response.content)

In [0]:
for endpt in metadata_endpts:
    get_endpoints(endpt,f"/Volumes/f1_warehouse/ingestion/raw_files/openf1/{endpt}.csv") #need to delete data before 2025 in silver layer, had trouble filtering>= 2025 in api

### Endpoints which require session key

In [0]:



# create delay once it hits api rate limit
def fetch_with_retry(url, params=None, max_retries=5, base_delay=1):  # dont need elif cos ur using if with return or continue
    for attempt in range(max_retries):
        response=requests.get(url, params=params)

        if response.status_code==200:  # 200 means its successful
            return response.json()

        if response.status_code==429: # rate limit error
            wait_time=response.headers.get("Retry-After")
            delay=int(wait_time) if wait_time else base_delay * (2 ** attempt)# if no retry after then use base delay
            print(f"hit rate limit, need to wait {delay}s (attempt {attempt+1}/{max_retries})")
            time.sleep(delay)  # part where program freezes for the delay time before retrying
            continue

        print(f"HTTP {response.status_code} for {url}, skipping")
        return None

    print(f"max retries exceeded for response code 429, skipping {url}")
    return None
    


      


             

        





In [0]:
#get session keys and filter for 2025 ownwards cos lesser data lesser loading time

sessions=spark.read.option("header","true").option("inferSchema","true").csv("/Volumes/f1_warehouse/ingestion/raw_files/openf1/sessions.csv")
sessions_filtered=sessions.filter(sessions.year>=2025) #filtered sessions for session keys

session_keys=[]
for row in sessions_filtered.select("session_key").distinct().collect():
    session_keys.append(row.session_key)

# u want session key and session type only
session_lookup=(sessions_filtered.filter(col("session_type").isin("Qualifying","Race","Sprint")).select("session_key","session_type").distinct().collect())

print(f"found {len(session_keys)} sessions (2025 ownwards)")
print(f"found {len(session_lookup)} qualifying/race/sprint sessions (2025 onwards)")


In [0]:
# loop through each session key in endpt and eventually get one csv for each endpoint

for endpt in session_endpts:
    all_records = []
    is_session_result = endpt == "session_result"

    if is_session_result:
        iterable = session_lookup
    else:
        iterable = session_keys  # pit, stints

    for item in iterable:

        if is_session_result:
            session_type = item.session_type
            session_key = item.session_key
        else:
            session_key = item

        records = fetch_with_retry(
            f"{BASE_URL}/{endpt}",
            params={"session_key": session_key}
        )

        if records:
            if is_session_result:
                for record in records:
                    # add session type to session_result
                    record["session_type"] = session_type

            all_records.extend(records)  # adds a whole list instead of just one item

        time.sleep(1)  # seconds between each request to prevent 429

    if all_records:
        df = pd.DataFrame(all_records)  # can use pandas cos less than mill rows
        output_path = f"/Volumes/f1_warehouse/ingestion/raw_files/openf1/{endpt}.csv"
        df.to_csv(output_path, index=False)
        print(f"{len(all_records)} for {endpt} added to {output_path}")